In [94]:
from scipy.sparse import kron, csc_matrix, eye
from itertools import combinations
from scipy.sparse.linalg import eigs, eigsh

class ConfigurationInteraction:
    def __init__(self, molecule: Molecule, molints: MolecularIntegrals):
        self.molecule = molecule
        self.molints = molints
        self.S = molints.overlap_matrix()
        self.T = molints.kinetic_matrix()
        self.V = molints.nuclear_attraction_matrix()
        self.ERI = molints.electron_repulsion_tensor(symmetrize = True)

        self.Nsite = 2 * self.S.shape[0]
        self.S_p = csc_matrix(np.array([[0, 0], [1, 0]], dtype=float))
        self.S_m = self.S_p.getH()
        self.Z = csc_matrix(np.array([[1, 0], [0, -1]], dtype=float))
        self.I = csc_matrix(np.array([[1, 0], [0, 1]], dtype=float))
        self.vacuum = eye(2**self.Nsite, format="csc")[:, 0]
        self.initialize_operators()

    def get_creation_operator(self, site):
        jw_string = csc_matrix(np.array([[1]], dtype=float))
        for i in range(site):
            jw_string = kron(jw_string, self.Z, format='csc')
        jw_string = kron(jw_string, self.S_p, format='csc')
        for i in range(site + 1, self.Nsite):
            jw_string = kron(jw_string, self.I, format='csc')
        return jw_string

    def get_creation_operators(self):
        a_dag = [self.get_creation_operator(site) for site in range(self.Nsite)]
        return a_dag
    
    def get_annihilation_operators(self):
        a = [op.getH() for op in self.get_creation_operators()]
        return a
    
    def initialize_operators(self):
        self.c = self.get_creation_operators()
        self.a = self.get_annihilation_operators()
    
    def get_number_operator(self, site):
        return self.c[site] @ self.a[site]
            
    def get_fullci_hamiltonian(self):
        eigvals, eigvecs = np.linalg.eigh(self.S)
        X = eigvecs @ np.diag(1/np.sqrt(eigvals))
        self.h = X.T @ (self.T+self.V) @ X 
        self.v = np.einsum('ijkl, ip,jq, kr,ls -> pqrs ', self.ERI, X,X,X,X)
        shape = self.c[0].shape   
        self.hamiltonian = 0.0*eye(*shape, format = 'csc')
        for i in range(self.Nsite):
            for j in range(self.Nsite):
                ij = (i%2 == j%2)
                self.hamiltonian +=  ij * self.h[i//2,j//2] * self.c[i] @ self.a[j]
                for k in range(self.Nsite):
                    for l in range(self.Nsite):
                        kl = (k%2 == l%2)

                        self.hamiltonian += 0.5 * ij * kl* self.v[i//2,j//2,k//2,l//2] * \
                        self.c[i] @ self.c[k] @ self.a[l] @ self.a[j]

        return self.hamiltonian
    
    def get_projection_operator(self, n_electrons):
        occupied_sites = list(combinations(range(self.Nsite), n_electrons))
        I_full = eye(2**self.Nsite, format = 'csc')
        P = 0.0*I_full
        for occ in occupied_sites:
            P_occ = 1.0*I_full
            for site in range(self.Nsite):
                if site in occ:
                    P_occ = P_occ @ self.get_number_operator(site)
                else:
                    P_occ = P_occ @ (I_full - self.get_number_operator(site))
            P += P_occ
        return P
    
    def get_projected_hamiltonian(self, n_electrons):
        H = self.get_fullci_hamiltonian()
        P = self.get_projection_operator(n_electrons)
        H_proj = P @ H @ P
        return 0.5 * (H_proj + H_proj.getH())
    
    def get_fullci_energies(self, n_electrons, n_roots=6):
        H_proj = self.get_projected_hamiltonian(n_electrons)
        dim = H_proj.shape[0]
        if dim == 1:
            return np.array([H_proj[0, 0]])
        n_roots = min(n_roots, dim - 1)
        if n_roots < 1:
            raise ValueError("n_roots must be at least 1")
        eigvals = eigsh(H_proj, k=n_roots, which='SA', return_eigenvectors=False)
        return np.sort(eigvals.real)
    
    def do_fullci(self, n_electrons):
        H_proj = self.get_projected_hamiltonian(n_electrons)
        eigvals, eigvecs = eigsh(H_proj, k=1, which='SA')
        return eigvals[0], eigvecs[:, 0]
            
                

In [95]:
import json
from pathlib import Path
from src_live import Molecule, BasisSet, MolecularIntegrals, ELEMENT_SYMBOL, Shell, S, T, V, ERI
import numpy as np

H4_xyz = """4
H4 molecule
H 1.0 1.0 0.0
H 1.0 -1.0 0.0
H -1.0 1.0 0.0
H -1.0 -1.0 0.0
"""
H4 = Molecule.from_string(H4_xyz)
sto3g = BasisSet("sto-3g")
basis_candidates = [Path("live_notebooks/sto-3g.json"), Path("sto-3g.json")]
for basis_path in basis_candidates:
    if basis_path.exists():
        sto3g.parse_elements(json.loads(basis_path.read_text()))
        break
else:
    sto3g.download_from_bse(["H"])
molints = MolecularIntegrals(H4, sto3g)
ERI_my = molints.electron_repulsion_tensor(symmetrize = True)


In [96]:
n_electrons = 4
n_roots = 6

ci = ConfigurationInteraction(H4, molints)
notebook_fci_roots_electronic = ci.get_fullci_energies(n_electrons=n_electrons, n_roots=n_roots)
notebook_fci_roots_electronic

array([-3.33038872, -3.31325788, -3.31325788, -3.31325788, -3.28994983,
       -3.28820774])

In [97]:
from pyscf import gto, scf, fci

pyscf_atoms = "; ".join(
    f"{atom.symbol} {atom.coord[0]} {atom.coord[1]} {atom.coord[2]}"
    for atom in H4.atoms
)
mol = gto.M(atom=pyscf_atoms, basis="sto-3g", unit="Angstrom")
mf = scf.RHF(mol)
mf.kernel()

pyscf_fci_total_roots, _ = fci.FCI(mf).kernel(nroots=n_roots)
pyscf_fci_total_roots = np.sort(np.asarray(pyscf_fci_total_roots))
pyscf_nuclear_repulsion = mol.energy_nuc()
pyscf_fci_electronic_roots = pyscf_fci_total_roots - pyscf_nuclear_repulsion

print("Notebook Full-CI electronic roots (fixed total electron number):")
print(f"{'Root':>4} {'Energy [Eh]':>22}")
for root, energy in enumerate(notebook_fci_roots_electronic):
    print(f"{root:>4d} {energy:22.12f}")

print()
print("PySCF Full-CI electronic roots ((N_alpha, N_beta) = (2, 2) sector):")
print(f"{'Root':>4} {'Energy [Eh]':>22}")
for root, energy in enumerate(pyscf_fci_electronic_roots):
    print(f"{root:>4d} {energy:22.12f}")

print()
print("Ground-state comparison:")
print(f"Notebook ground-state electronic [Eh]: {notebook_fci_roots_electronic[0]:.12f}")
print(f"PySCF ground-state electronic [Eh]:    {pyscf_fci_electronic_roots[0]:.12f}")
print(f"Difference [Eh]:                      {notebook_fci_roots_electronic[0] - pyscf_fci_electronic_roots[0]:.6e}")

print()
print(f"Nuclear repulsion [Eh]: {pyscf_nuclear_repulsion:.12f}")
print(f"Notebook ground-state total [Eh]: {notebook_fci_roots_electronic[0] + pyscf_nuclear_repulsion:.12f}")
print(f"PySCF ground-state total [Eh]:    {pyscf_fci_total_roots[0]:.12f}")
print()
print("Note: excited-state roots are not aligned one-to-one because the notebook projector fixes only total N, while the PySCF RHF-FCI solver works in the (N_alpha, N_beta) = (2, 2) sector.")


converged SCF energy = -1.54125526259767
Notebook Full-CI electronic roots (fixed total electron number):
Root            Energy [Eh]
   0        -3.330388723462
   1        -3.313257881940
   2        -3.313257881940
   3        -3.313257881940
   4        -3.289949830241
   5        -3.288207738630

PySCF Full-CI electronic roots ((N_alpha, N_beta) = (2, 2) sector):
Root            Energy [Eh]
   0        -3.330388605150
   1        -3.313257768908
   2        -3.289949727170
   3        -3.288207637092
   4        -3.288207637092
   5        -3.264923700663

Ground-state comparison:
Notebook ground-state electronic [Eh]: -3.330388723462
PySCF ground-state electronic [Eh]:    -3.330388605150
Difference [Eh]:                      -1.183119e-07

Nuclear repulsion [Eh]: 1.432539216131
Notebook ground-state total [Eh]: -1.897849507331
PySCF ground-state total [Eh]:    -1.897849389020

Note: excited-state roots are not aligned one-to-one because the notebook projector fixes only total N, 